<a href="https://colab.research.google.com/github/Vishali-Vivek006/Engineering-employability-prediction/blob/main/notebooks/06_resumeskill_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install -q pymupdf python-docx sentence-transformers pandas scikit-learn

In [8]:
from google.colab import files
import fitz
from docx import Document
import os, re, json
import pandas as pd

uploaded = files.upload()
resume_candidates = [name for name in uploaded if os.path.splitext(name)[1].lower() in {".pdf", ".docx"}]
if not resume_candidates:
    raise ValueError("Upload at least one PDF or DOCX resume.")
resume_filename = resume_candidates[0]
print("Resume selected:", resume_filename)
if "engineering_skill_catalogue.csv" in uploaded:
    print("Custom engineering skill catalogue uploaded; it will be used below.")

def extract_resume_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == ".pdf":
        with fitz.open(filepath) as pdf:
            text = "\n".join(page.get_text("text") for page in pdf)
    elif ext == ".docx":
        doc = Document(filepath)
        parts = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        text = "\n".join(parts)
    else:
        raise ValueError("Upload a PDF or DOCX resume.")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

resume_text = extract_resume_text(resume_filename)
print("Resume:", resume_filename)
print("Extracted characters:", len(resume_text))
print(resume_text[:8000])
if len(resume_text) < 50:
    print("WARNING: Very little text was extracted. If this is a scanned PDF, run OCR before continuing.")

Saving Vishali_Resume__1___1_ (3).pdf to Vishali_Resume__1___1_ (3).pdf
Resume selected: Vishali_Resume__1___1_ (3).pdf
Resume: Vishali_Resume__1___1_ (3).pdf
Extracted characters: 2093
VISHALI V
+91 6379742355
•
vishalivanith2006@gmail.com
LinkedIn: https://www.linkedin.com/in/vishali-vivekanandan-4876a42a4
GITHUB: https://github.com/Vishali-Vivek006
EDUCATION
• B.Tech Information Technology, MIT Campus Anna University, Chennai, Tamilnadu
Expected 2027
CGPA 8.55 (As of 6 semesters)
• Shepherds Matric.Hr.Sec.School,Tirupattur,Tamilnadu
Graduated 2023
Class 12: Secured 97.6%
SKILLS
Technical Skills
Algorithm and Data Structures, Database management, OOPS, Operating System.
Computer Networks,Machine Learning
Programming Languages
C, C++, Python, Java.
Database: SQL
Web Frameworks
Front-end: HTML, CSS, JS, React. Back-end: Node.js.
PROJECTS
Optimized Word Lookup System
• Built a command-line dictionary tool in C using a Trie data structure, enabling prefix-based word search with O(k)
look

In [9]:
# Robust profile extraction: education mentions, CGPA/GPA, percentages, and sections.
profile = {
    "education_mentions": [],
    "cgpa_mentions": [],
    "project_section": "",
    "internship_section": "",
    "certification_section": ""
}

education_patterns = [
    r"\bB\.?\s*Tech\b", r"\bB\.?\s*E\.?\b",
    r"\bM\.?\s*Tech\b", r"\bMCA\b", r"\bM\.?\s*Sc\.?\b",
    r"\bMBA\b", r"\bBachelor(?:'s)?\b", r"\bMaster(?:'s)?\b",
    r"\bB\.?\s*Sc\.?\b", r"\bB\.?\s*Com\.?\b"
]
for pattern in education_patterns:
    profile["education_mentions"].extend(
        re.findall(pattern, resume_text, flags=re.IGNORECASE)
    )
profile["education_mentions"] = sorted(set(profile["education_mentions"]))

# Handles e.g. CGPA: 8.52, CGPA 8.52/10, GPA - 3.8, 8.52 CGPA, GPA (4.0 scale): 3.6.
score_patterns = [
    ("label_before", re.compile(
        r"\b(?P<label>CGPA|CPI|GPA)\b\s*(?:\([^)]{0,30}\))?\s*[:=\-]?\s*(?P<score>\d{1,2}(?:\.\d{1,3})?)\s*(?:/\s*(?P<scale>10|4|100))?",
        re.IGNORECASE)),
    ("label_after", re.compile(
        r"(?P<score>\d{1,2}(?:\.\d{1,3})?)\s*(?:/\s*(?P<scale>10|4|100))?\s*\b(?P<label>CGPA|CPI|GPA)\b",
        re.IGNORECASE)),
]
score_matches = []
for kind, pattern in score_patterns:
    for match in pattern.finditer(resume_text):
        score = match.group("score")
        label = match.group("label").upper()
        scale = match.groupdict().get("scale")
        try:
            numeric_score = float(score)
        except ValueError:
            continue
        # Basic plausibility filter to avoid capturing years or unrelated numbers.
        if (scale == "10" and numeric_score <= 10) or (scale == "4" and numeric_score <= 4) or (scale == "100" and numeric_score <= 100):
            pass
        elif scale is None and numeric_score <= 10:
            pass
        else:
            continue
        score_matches.append({
            "label": label,
            "value": numeric_score,
            "scale": int(scale) if scale else None,
            "matched_text": match.group(0).strip(),
            "start": match.start(),
            "end": match.end()
        })
# Deduplicate overlapping matches, preserving first unique label/value/text combination.
seen = set()
profile["cgpa_mentions"] = []
for item in sorted(score_matches, key=lambda x: (x["start"], x["end"])):
    key = (item["label"], item["value"], item["scale"])
    if key not in seen:
        seen.add(key)
        profile["cgpa_mentions"].append(item)

# Section extraction uses exact normalized heading matching rather than substring matching.
def normalize_heading(line):
    line = re.sub(r"[^a-zA-Z &/+-]", " ", line).lower()
    return re.sub(r"\s+", " ", line).strip(" :.-")

def get_section(text, start_headings, end_headings):
    lines = text.splitlines()
    starts = {normalize_heading(x) for x in start_headings}
    ends = {normalize_heading(x) for x in end_headings}
    capture, collected = False, []
    for line in lines:
        clean = line.strip()
        if not clean:
            if capture:
                collected.append("")
            continue
        heading = normalize_heading(clean)
        # Accept heading variants like "Academic Projects" or "Work Experience".
        is_start = heading in starts or any(heading == h for h in starts)
        is_end = heading in ends
        if not capture and is_start:
            capture = True
            continue
        if capture and is_end:
            break
        if capture:
            collected.append(clean)
    return "\\n".join(collected).strip()

profile["project_section"] = get_section(
    resume_text,
    ["projects", "academic projects", "personal projects", "project experience"],
    ["education", "technical skills", "skills", "internship", "experience", "work experience", "certifications", "achievements", "publications"]
)
profile["internship_section"] = get_section(
    resume_text,
    ["internship", "internships", "experience", "work experience", "professional experience"],
    ["education", "technical skills", "skills", "projects", "certifications", "achievements", "publications"]
)
profile["certification_section"] = get_section(
    resume_text,
    ["certifications", "certificates", "licenses and certifications"],
    ["education", "technical skills", "skills", "projects", "internship", "experience", "achievements", "publications"]
)

print("Education mentions:", profile["education_mentions"])
print("CGPA/GPA matches:")
for item in profile["cgpa_mentions"]:
    print(" -", item["matched_text"], "=>", item["value"], "scale:", item["scale"])
print("Project section characters:", len(profile["project_section"]))
print("Internship section characters:", len(profile["internship_section"]))
print("Certification section characters:", len(profile["certification_section"]))

Education mentions: ['B.Tech']
CGPA/GPA matches:
 - CGPA 8.55 => 8.55 scale: None
Project section characters: 1456
Internship section characters: 0
Certification section characters: 143


In [10]:
# Starter catalogue. Add rows/aliases/descriptions to the CSV as your project grows.
ENGINEERING_SKILLS = {
    "CSE_IT": {
        "Programming": ["Python", "Java", "C++", "C#", "JavaScript", "TypeScript", "C", "R", "Go", "PHP", "Kotlin", "Ruby", "Swift", "Dart"],
        "Web_Development": ["HTML", "CSS", "React", "Angular", "Vue.js", "Node.js", "Express.js", "REST API", "Bootstrap"],
        "Database": ["SQL", "MySQL", "PostgreSQL", "MongoDB", "Oracle", "SQLite", "Redis"],
        "CS_Fundamentals": ["Data Structures", "Algorithms", "DSA", "OOP", "Operating Systems", "Computer Networks", "DBMS"],
        "AI_ML_Data": ["Machine Learning", "Deep Learning", "NumPy", "Pandas", "Matplotlib", "Scikit-learn", "TensorFlow", "PyTorch", "NLP", "Excel"],
        "Cloud_DevOps": ["AWS", "Azure", "Google Cloud", "Docker", "Kubernetes", "Git", "GitHub", "Linux", "Jenkins"]
    },
    "ECE": {
        "Electronics": ["Digital Electronics", "Analog Electronics", "Electronic Circuits", "Circuit Design", "VLSI", "CMOS", "Verilog", "VHDL", "FPGA", "PCB Design", "LTspice", "Cadence", "Proteus"],
        "Embedded_Systems": ["Embedded Systems", "Arduino", "Raspberry Pi", "Microcontrollers", "Microprocessors", "ARM", "8051", "Embedded C", "IoT"],
        "Signals_Communication": ["Digital Signal Processing", "Signal Processing", "Communication Systems", "Wireless Communication", "Antenna Design", "RF Engineering", "MATLAB", "Simulink", "Oscilloscope"]
    },
    "EEE": {
        "Electrical_Systems": ["Power Systems", "Electrical Machines", "Electrical Circuit Analysis", "Power Generation", "Power Transmission", "Power Distribution", "Electrical Protection", "Switchgear", "ETAP"],
        "Control_Power_Electronics": ["Power Electronics", "Control Systems", "PLC", "SCADA", "Industrial Automation", "Motor Control", "Drives", "MATLAB", "Simulink"],
        "Electrical_Design": ["AutoCAD Electrical", "Electrical Wiring", "Renewable Energy", "Solar PV", "Energy Management"]
    },
    "Mechanical": {
        "CAD_CAM": ["AutoCAD", "SolidWorks", "CATIA", "Creo", "Fusion 360", "NX CAD", "CAM", "CNC", "GD&T"],
        "Simulation_Analysis": ["ANSYS", "Finite Element Analysis", "FEA", "Computational Fluid Dynamics", "CFD", "Solid Mechanics", "Thermal Analysis", "Abaqus", "OpenFOAM"],
        "Core_Mechanical": ["Thermodynamics", "Fluid Mechanics", "Heat Transfer", "Machine Design", "Manufacturing Processes", "Robotics", "Materials Science", "Quality Control", "Six Sigma", "Lean Manufacturing"]
    },
    "Civil": {
        "Civil_Design": ["AutoCAD", "Revit", "STAAD.Pro", "ETABS", "Civil 3D", "BIM", "Structural Analysis", "Structural Design", "RCC Design", "Steel Structure Design"],
        "Construction": ["Quantity Surveying", "Estimation and Costing", "Project Planning", "Construction Management", "Primavera P6", "MS Project", "Site Supervision", "Quality Control"],
        "Core_Civil": ["Surveying", "Total Station", "Geotechnical Engineering", "Soil Mechanics", "Transportation Engineering", "Hydraulics", "Environmental Engineering"]
    },
    "Aerospace": {
        "Aircraft_Design": ["Aerodynamics", "Flight Mechanics", "Aircraft Structures", "Aircraft Design", "Propulsion", "Gas Turbines", "Orbital Mechanics", "Flight Dynamics"],
        "Simulation_Analysis": ["ANSYS", "CATIA", "MATLAB", "Simulink", "Computational Fluid Dynamics", "CFD", "Finite Element Analysis", "FEA", "Abaqus", "OpenFOAM"]
    }
}

catalogue_rows = []
for domain, categories in ENGINEERING_SKILLS.items():
    for category, skills in categories.items():
        for skill in skills:
            catalogue_rows.append({
                "skill": skill,
                "aliases": "",
                "domain": domain,
                "category": category,
                "description": f"{skill} in {category.replace('_', ' ')} for {domain.replace('_', '/')} engineering",
                "source": "Starter catalogue - verify and expand"
            })

starter_catalogue = pd.DataFrame(catalogue_rows).drop_duplicates(
    subset=["skill", "domain", "category"]
)

# If a previously expanded catalogue exists, use it rather than overwrite it.
if os.path.exists("engineering_skill_catalogue.csv"):
    catalogue_df = pd.read_csv("engineering_skill_catalogue.csv").fillna("")
    required_cols = ["skill", "aliases", "domain", "category"]
    missing_cols = [c for c in required_cols if c not in catalogue_df.columns]
    if missing_cols:
        raise ValueError(f"Catalogue is missing columns: {missing_cols}")
    if "description" not in catalogue_df.columns:
        catalogue_df["description"] = catalogue_df.apply(
            lambda r: f"{r['skill']} in {r['category']} for {r['domain']} engineering", axis=1
        )
    if "source" not in catalogue_df.columns:
        catalogue_df["source"] = "User catalogue"
else:
    catalogue_df = starter_catalogue
    catalogue_df.to_csv("engineering_skill_catalogue.csv", index=False)

print("Catalogue rows:", len(catalogue_df))
display(catalogue_df.groupby("domain")["skill"].nunique().reset_index(name="skills"))
print("Catalogue saved/loaded: engineering_skill_catalogue.csv")

Catalogue rows: 181


,domain,skills
0,Aerospace,18
1,CSE_IT,56
2,Civil,25
3,ECE,31
4,EEE,23
5,Mechanical,28


Catalogue saved/loaded: engineering_skill_catalogue.csv


In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- 1) Catalogue / alias matching ----------
def normalize_text(text):
    text = str(text).replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip().casefold()

def find_term(text, term):
    term_norm = normalize_text(term)
    if not term_norm:
        return False
    # Use word boundaries around the whole phrase to reduce partial-word matches.
    pattern = r"(?<!\w)" + re.escape(term_norm) + r"(?!\w)"
    return re.search(pattern, text) is not None

def extract_catalogue_matches(text, catalogue):
    normalized_resume = normalize_text(text)
    found = []
    for _, row in catalogue.iterrows():
        skill = str(row["skill"]).strip()
        if not skill:
            continue
        aliases = [x.strip() for x in str(row.get("aliases", "")).split(";") if x.strip()]
        matched_terms = [term for term in [skill] + aliases if find_term(normalized_resume, term)]
        if matched_terms:
            found.append({
                "skill": skill,
                "matched_text": matched_terms[0],
                "domain": row["domain"],
                "category": row["category"],
                "match_type": "catalogue_exact_or_alias",
                "similarity": None,
                "review_status": "matched text; proficiency not verified",
                "source": row.get("source", "")
            })
    cols = ["skill", "matched_text", "domain", "category", "match_type", "similarity", "review_status", "source"]
    result = pd.DataFrame(found, columns=cols)
    if not result.empty:
        result = result.drop_duplicates(subset=["skill", "domain", "category"])
    return result.reset_index(drop=True)

# ---------- 2) Semantic candidate matching ----------
def split_resume_sentences(text):
    # Resume bullets and lines often carry more useful evidence than the full resume embedding.
    pieces = re.split(r"[\n•●▪|]+|(?<=[.!?])\s+", text)
    sentences = [re.sub(r"\s+", " ", p).strip(" -\t") for p in pieces]
    return [s for s in sentences if len(s.split()) >= 3 and len(s) >= 15]

def semantic_skill_suggestions(text, catalogue, model, top_k_per_sentence=3, threshold=0.48):
    evidence = split_resume_sentences(text)
    if not evidence:
        return pd.DataFrame(columns=["skill", "matched_text", "domain", "category", "match_type", "similarity", "review_status", "source"])

    cat = catalogue.copy().reset_index(drop=True)
    descriptions = []
    for _, row in cat.iterrows():
        desc = str(row.get("description", "")).strip()
        if not desc:
            desc = f"{row['skill']}; category {row['category']}; domain {row['domain']} engineering"
        aliases = [x.strip() for x in str(row.get("aliases", "")).split(";") if x.strip()]
        descriptions.append(f"Skill: {row['skill']}. Aliases: {', '.join(aliases)}. Description: {desc}")

    evidence_emb = model.encode(evidence, normalize_embeddings=True, show_progress_bar=False)
    skill_emb = model.encode(descriptions, normalize_embeddings=True, show_progress_bar=False)
    sim = cosine_similarity(evidence_emb, skill_emb)

    rows = []
    for i, sentence in enumerate(evidence):
        ranked = sim[i].argsort()[::-1]
        added = 0
        for j in ranked:
            score = float(sim[i, j])
            if score < threshold:
                break
            row = cat.iloc[j]
            rows.append({
                "skill": row["skill"],
                "matched_text": sentence,
                "domain": row["domain"],
                "category": row["category"],
                "match_type": "semantic_candidate",
                "similarity": round(score, 3),
                "review_status": "needs human review; not confirmed",
                "source": row.get("source", "")
            })
            added += 1
            if added >= top_k_per_sentence:
                break

    cols = ["skill", "matched_text", "domain", "category", "match_type", "similarity", "review_status", "source"]
    result = pd.DataFrame(rows, columns=cols)
    if not result.empty:
        # Keep strongest semantic evidence per skill/domain/category.
        result = result.sort_values("similarity", ascending=False).drop_duplicates(
            subset=["skill", "domain", "category"]
        )
    return result.reset_index(drop=True)

# Load model once. First run downloads the model.
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

catalogue_matches = extract_catalogue_matches(resume_text, catalogue_df)
semantic_candidates = semantic_skill_suggestions(
    resume_text,
    catalogue_df,
    embedding_model,
    top_k_per_sentence=3,
    threshold=0.48  # Tune using labelled resume examples; not a universal cutoff.
)

# Do not silently merge semantic candidates into confirmed skills.
confirmed_keys = set(
    (r.skill.casefold(), r.domain, r.category)
    for r in catalogue_matches.itertuples()
)
if not semantic_candidates.empty:
    semantic_candidates = semantic_candidates[
        ~semantic_candidates.apply(
            lambda r: (r["skill"].casefold(), r["domain"], r["category"]) in confirmed_keys,
            axis=1
        )
    ].reset_index(drop=True)

print("Catalogue matches:", len(catalogue_matches))
print("Semantic candidates for review:", len(semantic_candidates))
print("\\nConfirmed catalogue matches:")
display(catalogue_matches)
print("\\nSemantic suggestions — review before accepting:")
display(semantic_candidates)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Catalogue matches: 16
Semantic candidates for review: 3
\nConfirmed catalogue matches:


,skill,matched_text,domain,category,match_type,similarity,review_status,source
0,Python,Python,CSE_IT,Programming,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
1,Java,Java,CSE_IT,Programming,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
2,C++,C++,CSE_IT,Programming,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
3,C,C,CSE_IT,Programming,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
4,HTML,HTML,CSE_IT,Web_Development,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
5,CSS,CSS,CSE_IT,Web_Development,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
6,React,React,CSE_IT,Web_Development,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
7,Node.js,Node.js,CSE_IT,Web_Development,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
8,SQL,SQL,CSE_IT,Database,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand
9,MySQL,MySQL,CSE_IT,Database,catalogue_exact_or_alias,None,matched text; proficiency not verified,Starter catalogue - verify and expand


\nSemantic suggestions — review before accepting:


,skill,matched_text,domain,category,match_type,similarity,review_status,source
0,NumPy,Python for Data Science- NPTEL,CSE_IT,AI_ML_Data,semantic_candidate,0.508,needs human review; not confirmed,Starter catalogue - verify and expand
1,Express.js,"Front-end: HTML, CSS, JS, React.",CSE_IT,Web_Development,semantic_candidate,0.489,needs human review; not confirmed,Starter catalogue - verify and expand
2,Scikit-learn,"Technologies involved: Python, Machine Learning.",CSE_IT,AI_ML_Data,semantic_candidate,0.489,needs human review; not confirmed,Starter catalogue - verify and expand


In [12]:
# Save a consistent output from the updated hybrid extractor.
all_skill_rows = pd.concat(
    [catalogue_matches, semantic_candidates],
    ignore_index=True
).drop_duplicates(
    subset=["skill", "domain", "category", "match_type"]
)

# Only catalogue matches count as matched/extracted skills by default.
confirmed_skills = sorted(catalogue_matches["skill"].dropna().unique().tolist()) if not catalogue_matches.empty else []
semantic_suggestions = semantic_candidates.to_dict(orient="records") if not semantic_candidates.empty else []

output = {
    "resume_filename": resume_filename,
    "education_mentions": profile["education_mentions"],
    "cgpa_mentions": profile["cgpa_mentions"],
    "technical_skills": confirmed_skills,
    "skills_by_domain": {},
    "catalogue_skill_matches": catalogue_matches.to_dict(orient="records"),
    "semantic_skill_suggestions": semantic_suggestions,
    "project_section": profile["project_section"],
    "internship_section": profile["internship_section"],
    "certification_section": profile["certification_section"]
}

if not catalogue_matches.empty:
    for domain, group in catalogue_matches.groupby("domain"):
        output["skills_by_domain"][domain] = {}
        for category, cat_group in group.groupby("category"):
            output["skills_by_domain"][domain][category] = sorted(cat_group["skill"].unique().tolist())

with open("resume_extraction_output.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

all_skill_rows.to_csv("extracted_skills.csv", index=False)
catalogue_df.to_csv("engineering_skill_catalogue.csv", index=False)

print("Saved resume_extraction_output.json")
print("Saved extracted_skills.csv")
print("Saved engineering_skill_catalogue.csv")
print("Confirmed catalogue skills:", len(confirmed_skills))
print("Semantic suggestions for review:", len(semantic_suggestions))

Saved resume_extraction_output.json
Saved extracted_skills.csv
Saved engineering_skill_catalogue.csv
Confirmed catalogue skills: 16
Semantic suggestions for review: 3


/tmp/ipykernel_1931/2741126292.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_skill_rows = pd.concat(
